<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# Machine Learning Foundation

## Course 5, Part g: Transfer Learning DEMO


For this exercise, we will use the well-known MNIST digit data. To illustrate the power and concept of transfer learning, we will train a CNN on just the digits 5,6,7,8,9.  Then we will train just the last layer(s) of the network on the digits 0,1,2,3,4 and see how well the features learned on 5-9 help with classifying 0-4.




In [1]:
import datetime
import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras import backend as K
#from tensorflow import keras
#from tensorflow.keras.datasets import mnist
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
#from tensorflow.keras.layers import Conv2D, MaxPooling2D
#from tensorflow.keras import backend as K

Using TensorFlow backend.
/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/jupyterlab/con

In [2]:
#used to help some of the timing functions
now = datetime.datetime.now

In [3]:
# set some parameters
batch_size = 128
num_classes = 5
epochs = 5

In [4]:
# set some more parameters
img_rows, img_cols = 28, 28
filters = 32
pool_size = 2
kernel_size = 3

In [5]:
## This just handles some variability in how the input data is loaded

if K.image_data_format() == 'channels_first':
    input_shape = (1, img_rows, img_cols)
else:
    input_shape = (img_rows, img_cols, 1)

In [6]:
## To simplify things, write a function to include all the training steps
## As input, function takes a model, training set, test set, and the number of classes
## Inside the model object will be the state about which layers we are freezing and which we are training

def train_model(model, train, test, num_classes):
    x_train = train[0].reshape((train[0].shape[0],) + input_shape)
    x_test = test[0].reshape((test[0].shape[0],) + input_shape)
    x_train = x_train.astype('float32')
    x_test = x_test.astype('float32')
    x_train /= 255
    x_test /= 255
    print('x_train shape:', x_train.shape)
    print(x_train.shape[0], 'train samples')
    print(x_test.shape[0], 'test samples')

    # convert class vectors to binary class matrices
    y_train = keras.utils.to_categorical(train[1], num_classes)
    y_test = keras.utils.to_categorical(test[1], num_classes)

    model.compile(loss='categorical_crossentropy',
                  optimizer='adadelta',
                  metrics=['accuracy'])

    t = now()
    model.fit(x_train, y_train,
              batch_size=batch_size,
              epochs=epochs,
              verbose=1,
              validation_data=(x_test, y_test))
    print('Training time: %s' % (now() - t))

    score = model.evaluate(x_test, y_test, verbose=0)
    print('Test score:', score[0])
    print('Test accuracy:', score[1])

这段代码将所有的训练步骤封装到一个函数 `train_model` 中，只需传入不同的模型实例及数据，就能完成数据预处理、模型编译、训练和评估的全流程。下面按块解释它在做什么：

```python
def train_model(model, train, test, num_classes):
```
- **输入**  
  - `model`：一个已构造好的 Keras 模型对象，它内部包含了哪些层要冻结、哪些层要训练的设置（如在迁移学习中常见）。  
  - `train`：训练集，格式为 `(x_train_array, y_train_array)`。  
  - `test`：测试集，格式为 `(x_test_array, y_test_array)`。  
  - `num_classes`：分类任务的类别数，用于 one-hot 编码。

---

### 1. 处理输入数据形状与归一化

```python
x_train = train[0].reshape((train[0].shape[0],) + input_shape)
x_test  = test[0].reshape((test[0].shape[0],)  + input_shape)

x_train = x_train.astype('float32')
x_test  = x_test.astype('float32')

x_train /= 255
x_test  /= 255

print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')
```
1. **`reshape`**：将扁平化的像素数组重塑回 `(样本数, 高, 宽, 通道数)`，这里 `input_shape` 可能是 `(height, width, channels)`。  
2. **`astype('float32')`**：转成 32 位浮点，符合大多数深度学习库的数值格式要求。  
3. **归一化**：除以 255，把像素值从 `[0, 255]` 映射到 `[0, 1]`，加快收敛并提高数值稳定性。  
4. **打印**：输出训练/测试集的最终形状和样本数，便于调试和日志记录。

---

### 2. 标签的 One-Hot 编码

```python
y_train = keras.utils.to_categorical(train[1], num_classes)
y_test  = keras.utils.to_categorical(test[1],  num_classes)
```
- 把整数标签（`0` 到 `num_classes-1`）转换成 one-hot 矩阵，例如类别 `3` → `[0,0,0,1,0,…]`。  
- 为后续使用 `categorical_crossentropy` 损失函数做准备。

---

### 3. 模型编译

```python
model.compile(
    loss='categorical_crossentropy',
    optimizer='adadelta',
    metrics=['accuracy']
)
```
- **损失函数**：`categorical_crossentropy`，适合多类分类的 one-hot 标签。  
- **优化器**：`Adadelta`，一种自适应学习率的优化算法，无需手动设置初始学习率。  
- **评估指标**：`accuracy`，在训练和验证时显示分类准确率。

---

### 4. 模型训练并计时

```python
t = now()
model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    validation_data=(x_test, y_test)
)
print('Training time: %s' % (now() - t))
```
- **`now()`**：用户自定义的获取当前时间函数，用于计算训练耗时。  
- **`fit`**：  
  - 批大小 `batch_size`，每次梯度更新使用的样本数。  
  - 训练轮数 `epochs`。  
  - `validation_data`：在每个 epoch 结束后，用测试集评估一次并打印验证损失和准确率。  
- **打印训练时长**，帮助评估性能或对比不同配置。

---

### 5. 模型评估

```python
score = model.evaluate(x_test, y_test, verbose=0)
print('Test score:',    score[0])
print('Test accuracy:', score[1])
```
- **`evaluate`**：在测试集上做一次完整的前向传播，返回 `[loss, accuracy]`。  
- 最后打印测试集上的损失 (`score[0]`) 和准确率 (`score[1]`)。

---

**总结**：  
- 这个函数自动化了数据预处理（reshape、归一化、one-hot 编码）、模型编译、训练（含验证）和最终评估的步骤；  
- 只需传入不同的 `model`、`train`、`test` 和 `num_classes`，即可重复地训练、比较和记录多种模型或超参数配置。

In [7]:
# the data, shuffled and split between train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# create two datasets: one with digits below 5 and one with 5 and above
x_train_lt5 = x_train[y_train < 5]
y_train_lt5 = y_train[y_train < 5]
x_test_lt5 = x_test[y_test < 5]
y_test_lt5 = y_test[y_test < 5]

x_train_gte5 = x_train[y_train >= 5]
y_train_gte5 = y_train[y_train >= 5] - 5
x_test_gte5 = x_test[y_test >= 5]
y_test_gte5 = y_test[y_test >= 5] - 5

11493376/11490434 [==============================] - 0s 0us/step


In [8]:
# Define the "feature" layers.  These are the early layers that we expect will "transfer"
# to a new problem.  We will freeze these layers during the fine-tuning process

feature_layers = [
    Conv2D(filters, kernel_size,
           padding='valid',
           input_shape=input_shape),
    Activation('relu'),
    Conv2D(filters, kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size=pool_size),
    Dropout(0.25),
    Flatten(),
]

In [9]:
# Define the "classification" layers.  These are the later layers that predict the specific classes from the features
# learned by the feature layers.  This is the part of the model that needs to be re-trained for a new problem

classification_layers = [
    Dense(128),
    Activation('relu'),
    Dropout(0.5),
    Dense(num_classes),
    Activation('softmax')
]

In [10]:
# We create our model by combining the two sets of layers as follows
model = Sequential(feature_layers + classification_layers)





Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [11]:
# Let's take a look
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
activation_1 (Activation)    (None, 26, 26, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 24, 24, 32)        9248      
_________________________________________________________________
activation_2 (Activation)    (None, 24, 24, 32)        0         
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 12, 12, 32)        0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 12, 12, 32)        0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 4608)              0         
__________

In [12]:
# Now, let's train our model on the digits 5,6,7,8,9

train_model(model,
            (x_train_gte5, y_train_gte5),
            (x_test_gte5, y_test_gte5), num_classes)

x_train shape: (29404, 28, 28, 1)
29404 train samples
4861 test samples


Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Train on 29404 samples, validate on 4861 samples
Epoch 1/5


2025-04-27 08:56:46.338820: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
2025-04-27 08:56:46.345375: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2394315000 Hz
2025-04-27 08:56:46.345989: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5556b8a45b40 executing computations on platform Host. Devices:
2025-04-27 08:56:46.346035: I tensorflow/compiler/xla/service/service.cc:175]   StreamExecutor device (0): <undefined>, <undefined>
2025-04-27 08:56:46.472090: W tensorflow/compiler/jit/mark_for_compilation_pass.cc:1412] (One-time warning): Not using XLA:CPU for cluster because envvar TF_XLA_FLAGS=--tf_xla_cpu_global_jit was not set.  If you want XLA:CPU, either set that envvar, or use experimental_jit_scope to enable XLA:CPU.  To confirm that XLA is active, pass --vmodule=xla_compilation_cache=1 (as a proper command-line fl

29404/29404 [==============================] - 92s 3ms/step - loss: 0.2269 - acc: 0.9241 - val_loss: 0.0466 - val_acc: 0.9852
Epoch 2/5
29404/29404 [==============================] - 89s 3ms/step - loss: 0.0614 - acc: 0.9810 - val_loss: 0.0418 - val_acc: 0.9848
Epoch 3/5
29404/29404 [==============================] - 89s 3ms/step - loss: 0.0482 - acc: 0.9855 - val_loss: 0.0314 - val_acc: 0.9887
Epoch 4/5
29404/29404 [==============================] - 88s 3ms/step - loss: 0.0367 - acc: 0.9882 - val_loss: 0.0243 - val_acc: 0.9924
Epoch 5/5
29404/29404 [==============================] - 87s 3ms/step - loss: 0.0312 - acc: 0.9903 - val_loss: 0.0215 - val_acc: 0.9936
Training time: 0:07:24.771564
Test score: 0.02145871215923856
Test accuracy: 0.9936227112659037


### Freezing Layers
Keras allows layers to be "frozen" during the training process.  That is, some layers would have their weights updated during the training process, while others would not.  This is a core part of transfer learning, the ability to train just the last one or several layers.

Note also, that a lot of the training time is spent "back-propagating" the gradients back to the first layer.  Therefore, if we only need to compute the gradients back a small number of layers, the training time is much quicker per iteration.  This is in addition to the savings gained by being able to train on a smaller data set.

### 冻结层

Keras 允许在训练过程中“冻结”某些层。也就是说，有些层的权重在训练时会更新，而另一些层则保持不变。这是迁移学习的核心——仅训练最后一层或几层的能力。

另外，请注意，大量的训练时间都花在将梯度“反向传播”到第一层上。因此，如果我们只需计算并传播到少数几层，那么每次迭代的训练时间就会大大缩短。这还不包括因为只在更小的数据集上训练而节省的时间。

In [13]:
# Freeze only the feature layers
for l in feature_layers:
    l.trainable = False

Observe below the differences between the number of *total params*, *trainable params*, and *non-trainable params*.


In [14]:
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
activation_1 (Activation)    (None, 26, 26, 32)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 24, 24, 32)        9248      
_________________________________________________________________
activation_2 (Activation)    (None, 24, 24, 32)        0         
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 12, 12, 32)        0         
_________________________________________________________________
dropout_1 (Dropout)          (None, 12, 12, 32)        0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 4608)              0         
__________

In [15]:
train_model(model,
            (x_train_lt5, y_train_lt5),
            (x_test_lt5, y_test_lt5), num_classes)

x_train shape: (30596, 28, 28, 1)
30596 train samples
5139 test samples
Train on 30596 samples, validate on 5139 samples
Epoch 1/5
30596/30596 [==============================] - 34s 1ms/step - loss: 0.1602 - acc: 0.9565 - val_loss: 0.0190 - val_acc: 0.9940
Epoch 2/5
30596/30596 [==============================] - 32s 1ms/step - loss: 0.0415 - acc: 0.9878 - val_loss: 0.0117 - val_acc: 0.9963
Epoch 3/5
30596/30596 [==============================] - 34s 1ms/step - loss: 0.0298 - acc: 0.9914 - val_loss: 0.0104 - val_acc: 0.9965
Epoch 4/5
30596/30596 [==============================] - 33s 1ms/step - loss: 0.0246 - acc: 0.9922 - val_loss: 0.0088 - val_acc: 0.9977
Epoch 5/5
30596/30596 [==============================] - 34s 1ms/step - loss: 0.0191 - acc: 0.9942 - val_loss: 0.0065 - val_acc: 0.9981
Training time: 0:02:47.466004
Test score: 0.006513473929332778
Test accuracy: 0.9980540961276513


Note that after a single epoch, we are already achieving results on classifying 0-4 that are comparable to those achieved on 5-9 after 5 full epochs.  This despite the fact the we are only "fine-tuning" the last layer of the network, and all the early layers have never seen what the digits 0-4 look like.

Also, note that even though nearly all (590K/600K) of the *parameters* were trainable, the training time per epoch was still much reduced.  This is because the unfrozen part of the network was very shallow, making backpropagation faster. 


## Exercise
- Now we will write code to reverse this training process.  That is, train on the digits 0-4, then finetune only the last layers on the digits 5-9.


In [16]:
# Create layers and define the model as above
feature_layers2 = [
    Conv2D(filters, kernel_size,
           padding='valid',
           input_shape=input_shape),
    Activation('relu'),
    Conv2D(filters, kernel_size),
    Activation('relu'),
    MaxPooling2D(pool_size=pool_size),
    Dropout(0.25),
    Flatten(),
]

classification_layers2 = [
    Dense(128),
    Activation('relu'),
    Dropout(0.5),
    Dense(num_classes),
    Activation('softmax')
]
model2 = Sequential(feature_layers2 + classification_layers2)
model2.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_3 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
activation_5 (Activation)    (None, 26, 26, 32)        0         
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 24, 24, 32)        9248      
_________________________________________________________________
activation_6 (Activation)    (None, 24, 24, 32)        0         
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 12, 12, 32)        0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 12, 12, 32)        0         
_________________________________________________________________
flatten_2 (Flatten)          (None, 4608)              0         
__________

In [17]:
# Now, let's train our model on the digits 0,1,2,3,4
train_model(model2,
            (x_train_lt5, y_train_lt5),
            (x_test_lt5, y_test_lt5), num_classes)

x_train shape: (30596, 28, 28, 1)
30596 train samples
5139 test samples
Train on 30596 samples, validate on 5139 samples
Epoch 1/5
30596/30596 [==============================] - 90s 3ms/step - loss: 0.1706 - acc: 0.9461 - val_loss: 0.0288 - val_acc: 0.9899
Epoch 2/5
30596/30596 [==============================] - 90s 3ms/step - loss: 0.0488 - acc: 0.9853 - val_loss: 0.0268 - val_acc: 0.9901
Epoch 3/5
30596/30596 [==============================] - 89s 3ms/step - loss: 0.0328 - acc: 0.9896 - val_loss: 0.0118 - val_acc: 0.9953
Epoch 4/5
30596/30596 [==============================] - 92s 3ms/step - loss: 0.0253 - acc: 0.9926 - val_loss: 0.0085 - val_acc: 0.9965
Epoch 5/5
30596/30596 [==============================] - 92s 3ms/step - loss: 0.0235 - acc: 0.9927 - val_loss: 0.0072 - val_acc: 0.9975
Training time: 0:07:33.480551
Test score: 0.007187740243145295
Test accuracy: 0.9974703249659467


In [18]:
#Freeze layers
for l in feature_layers2:
    l.trainable = False

In [19]:
model2.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_3 (Conv2D)            (None, 26, 26, 32)        320       
_________________________________________________________________
activation_5 (Activation)    (None, 26, 26, 32)        0         
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 24, 24, 32)        9248      
_________________________________________________________________
activation_6 (Activation)    (None, 24, 24, 32)        0         
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 12, 12, 32)        0         
_________________________________________________________________
dropout_3 (Dropout)          (None, 12, 12, 32)        0         
_________________________________________________________________
flatten_2 (Flatten)          (None, 4608)              0         
__________

In [20]:
train_model(model2,
            (x_train_gte5, y_train_gte5),
            (x_test_gte5, y_test_gte5), num_classes)

x_train shape: (29404, 28, 28, 1)
29404 train samples
4861 test samples
Train on 29404 samples, validate on 4861 samples
Epoch 1/5
29404/29404 [==============================] - 31s 1ms/step - loss: 0.2326 - acc: 0.9323 - val_loss: 0.0503 - val_acc: 0.9850
Epoch 2/5
29404/29404 [==============================] - 31s 1ms/step - loss: 0.0751 - acc: 0.9770 - val_loss: 0.0346 - val_acc: 0.9883
Epoch 3/5
29404/29404 [==============================] - 31s 1ms/step - loss: 0.0593 - acc: 0.9825 - val_loss: 0.0287 - val_acc: 0.9889
Epoch 4/5
29404/29404 [==============================] - 33s 1ms/step - loss: 0.0523 - acc: 0.9843 - val_loss: 0.0259 - val_acc: 0.9907
Epoch 5/5
29404/29404 [==============================] - 32s 1ms/step - loss: 0.0434 - acc: 0.9871 - val_loss: 0.0242 - val_acc: 0.9920
Training time: 0:02:38.296509
Test score: 0.024229986986726473
Test accuracy: 0.9919769594733594


---
### Machine Learning Foundation (C) 2020 IBM Corporation
